In [3]:
# graphrag_plus.py

import json
import pandas as pd
import networkx as nx

from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

import ollama



In [7]:
# Load FAQ (already generated from refined.csv using your script)
faq = pd.read_csv("/Users/iramkamdar/RAG/data/faq_updated.csv", encoding="utf-8")

# Load merged JSON with email content (prospect_email and reply)
with open("/Users/iramkamdar/RAG/data/generated_email_pairs.json", "r", encoding="utf-8") as f:
    labels = json.load(f)

print(f"FAQ rows: {len(faq)}")
print(f"Labelled items: {len(labels)}")
print(f"Sample item keys: {list(labels[0].keys()) if labels else 'No data'}")


FAQ rows: 22
Labelled items: 104
Sample item keys: ['id', 'labels', 'subject', 'sender_email', 'prospect_email', 'reply']


In [8]:
# Extract all unique intents from merged data
unique_intents = set()
for item in labels:
    for intent in item.get("labels", {}).get("intents", []):
        unique_intents.add(intent)

print("Unique Intents:", unique_intents)


Unique Intents: {'confirm', 'accept_or_decline', 'schedule', 'send_materials', 'request_info', 'reschedule', 'share_feedback', 'request_feedback'}


In [9]:
# Preview merged data structure
for i, item in enumerate(labels[:3]):
    print(f"\n=== Item {i+1} ===")
    print(f"ID: {item.get('id')}")
    print(f"Subject: {item.get('subject')}")
    print(f"Has prospect_email: {'prospect_email' in item and item['prospect_email'] is not None}")
    print(f"Has reply: {'reply' in item and item['reply'] is not None}")
    print(f"Labels: {item.get('labels')}")
    if item.get('prospect_email'):
        print(f"Prospect email preview: {item['prospect_email'][:100]}...")



=== Item 1 ===
ID: eef2f02b-5d07-45dd-9098-b4fe094cfd29
Subject: Meeting to Discuss Research Collaboration
Has prospect_email: True
Has reply: True
Labels: {'topic': 'Professor/Academic', 'intents': ['schedule'], 'artifacts': ['calendly']}
Prospect email preview: Hi Zubair, I hope you're well. Could we schedule a time to discuss the upcoming research project? Pl...

=== Item 2 ===
ID: 24944f88-16fb-4b0e-87e4-dba71859c779
Subject: Thesis Guidance
Has prospect_email: True
Has reply: True
Labels: {'topic': 'Professor/Academic', 'intents': ['schedule'], 'artifacts': ['calendly']}
Prospect email preview: Hi Zubair, I am reaching out to schedule a meeting for thesis guidance. Can we find a time this week...

=== Item 3 ===
ID: a99f3889-0b49-4dd5-93c5-ccc1778ff152
Subject: Project Update
Has prospect_email: True
Has reply: True
Labels: {'topic': 'Professor/Academic', 'intents': ['schedule'], 'artifacts': ['calendly']}
Prospect email preview: Hi Zubair, I wanted to touch base on the project w

In [10]:
import networkx as nx

G = nx.DiGraph()

# Store email content mapping for each node
# This will help us include personalized email context in embeddings
email_content_map = {}  # {node_name: [list of email contexts]}

for item in labels:
    label_data = item.get("labels", {})
    topic = label_data.get("topic")
    intents = label_data.get("intents", [])
    artifacts = label_data.get("artifacts", [])
    
    # Get email content for personalization
    prospect_email = item.get("prospect_email", "")
    reply = item.get("reply", "")
    subject = item.get("subject", "")
    email_id = item.get("id", "")
    
    # Create email context string
    email_context = f"Subject: {subject}\nEmail: {prospect_email[:200] if prospect_email else 'N/A'}\nReply: {reply[:200] if reply else 'N/A'}"

    if not topic and not intents and not artifacts:
        continue

    # Add topic node with email context
    if topic:
        G.add_node(topic, type="topic")
        if topic not in email_content_map:
            email_content_map[topic] = []
        email_content_map[topic].append({
            "email_id": email_id,
            "subject": subject,
            "prospect_email": prospect_email[:300] if prospect_email else "",
            "reply": reply[:300] if reply else ""
        })

    # Add intents as nodes and edges with email context
    for intent in intents:
        G.add_node(intent, type="intent")
        if topic:
            G.add_edge(topic, intent, relation="HAS_INTENT", email_id=email_id)
        
        # Store email context for intent nodes
        if intent not in email_content_map:
            email_content_map[intent] = []
        email_content_map[intent].append({
            "email_id": email_id,
            "subject": subject,
            "prospect_email": prospect_email[:300] if prospect_email else "",
            "reply": reply[:300] if reply else ""
        })

    # Add artifacts as nodes and edges with email context
    for artifact in artifacts:
        G.add_node(artifact, type="artifact")
        if topic:
            G.add_edge(topic, artifact, relation="USES_ARTIFACT", email_id=email_id)
        
        # Store email context for artifact nodes
        if artifact not in email_content_map:
            email_content_map[artifact] = []
        email_content_map[artifact].append({
            "email_id": email_id,
            "subject": subject,
            "prospect_email": prospect_email[:300] if prospect_email else "",
            "reply": reply[:300] if reply else ""
        })

print(f"Graph: {len(G.nodes())} nodes, {len(G.edges())} edges")
print(f"Nodes with email context: {len([n for n in G.nodes() if n in email_content_map])}")
print(f"Sample node with context: {list(email_content_map.keys())[0] if email_content_map else 'None'}")


Graph: 23 nodes, 38 edges
Nodes with email context: 23
Sample node with context: Professor/Academic


In [11]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

qdrant = QdrantClient(path="qdrant_data")   # persistent local storage

qdrant.recreate_collection(
    collection_name="knowledge_space",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)


True

## if the above cell doesn't work , RUN rm -f qdrant_data/.lock      

In [12]:
faq_texts = [
    f"FAQ | Question: {row['question']} | Answer: {row['answer']}"
    for _, row in faq.iterrows()
]
faq_vectors = embedder.encode(faq_texts, show_progress_bar=False)
faq_payloads = []

for idx, row in faq.iterrows():
    faq_payloads.append({
        "type": "faq",
        "id_kind": "faq",
        "faq_id": int(idx),
        "question": row["question"], 
        "answer": row["answer"],
    })


In [13]:
graph_nodes = list(G.nodes(data=True))  # [(name, attrs), ...]

graph_texts = []
graph_payloads = []

for i, (name, attrs) in enumerate(graph_nodes):
    ntype = attrs.get("type", "unknown")
    neighbors = list(G.successors(name)) + list(G.predecessors(name))
    neighbors_str = ", ".join(neighbors) if neighbors else "None"
    
    # Get email context for this node (personalized content)
    email_contexts = email_content_map.get(name, [])
    
    # Build enriched text with email content for better personalization
    email_examples = ""
    if email_contexts:
        # Include up to 2 email examples for context
        for ctx in email_contexts[:2]:
            if ctx.get("prospect_email"):
                email_examples += f"\nExample Email: {ctx['prospect_email'][:150]}"
            if ctx.get("reply"):
                email_examples += f"\nExample Reply: {ctx['reply'][:150]}"
    
    # Enhanced text with email context
    text = f"GRAPH_NODE | Type: {ntype} | Name: {name} | Neighbors: {neighbors_str}{email_examples}"
    graph_texts.append(text)

    # Enhanced payload with email content
    graph_payloads.append({
        "type": "graph_node",
        "id_kind": "graph_node",
        "node_name": name,
        "node_type": ntype,
        "neighbors": neighbors,
        "email_contexts": email_contexts[:3],  # Store up to 3 email examples
        "email_count": len(email_contexts),  # Total number of emails associated
    })

graph_vectors = embedder.encode(graph_texts, show_progress_bar=False)
print(f"✅ Encoded {len(graph_vectors)} graph nodes with personalized email content")


✅ Encoded 23 graph nodes with personalized email content


In [14]:
# GMAIL-STYLE: Create separate collection for writing style retrieval
# This indexes reply chunks (200-400 tokens) for semantic style matching

print("📝 Creating writing style collection...")

# Step 1: Extract all reply chunks (200-400 tokens ≈ 150-300 words)
style_chunks = []
chunk_size_words = 200  # words per chunk (approximately 200-400 tokens)

for item in labels:
    reply = item.get("reply", "")
    if reply and len(reply.strip()) > 50:  # Only process non-empty replies
        # Break reply into word chunks
        words = reply.split()
        
        # If reply is shorter than chunk_size, use whole reply
        if len(words) <= chunk_size_words:
            style_chunks.append({
                "chunk_id": f"{item['id']}_chunk_0",
                "email_id": item["id"],
                "subject": item.get("subject", ""),
                "reply_chunk": reply,  # ✅ STYLE TEXT ONLY
                "intent": item.get("labels", {}).get("intents", [""])[0] if item.get("labels", {}).get("intents") else "",
                "type": "writing_style"  # ✅ Mark as style, not content
            })
        else:
            # Split into multiple chunks
            for i in range(0, len(words), chunk_size_words):
                chunk_words = words[i:i+chunk_size_words]
                chunk_text = " ".join(chunk_words)
                style_chunks.append({
                    "chunk_id": f"{item['id']}_chunk_{i//chunk_size_words}",
                    "email_id": item["id"],
                    "subject": item.get("subject", ""),
                    "reply_chunk": chunk_text,  # ✅ STYLE TEXT ONLY
                    "intent": item.get("labels", {}).get("intents", [""])[0] if item.get("labels", {}).get("intents") else "",
                    "type": "writing_style"
                })

print(f"✅ Extracted {len(style_chunks)} writing style chunks from {len([l for l in labels if l.get('reply')])} replies")

# Step 2: Embed style chunks
style_texts = [chunk["reply_chunk"] for chunk in style_chunks]
style_vectors = embedder.encode(style_texts, show_progress_bar=False)

# Step 3: Create separate Qdrant collection for style
try:
    qdrant.delete_collection(collection_name="writing_style")
except:
    pass

qdrant.create_collection(
    collection_name="writing_style",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

# Step 4: Store style chunks in Qdrant
style_points = [
    PointStruct(
        id=i,
        vector=vec.tolist(),
        payload=chunk
    )
    for i, (vec, chunk) in enumerate(zip(style_vectors, style_chunks))
]

qdrant.upsert(collection_name="writing_style", points=style_points)
print(f"✅ Indexed {len(style_chunks)} writing style chunks in 'writing_style' collection")
print(f"   Average chunk length: {sum(len(c['reply_chunk'].split()) for c in style_chunks) // len(style_chunks) if style_chunks else 0} words")


📝 Creating writing style collection...
✅ Extracted 104 writing style chunks from 104 replies
✅ Indexed 104 writing style chunks in 'writing_style' collection
   Average chunk length: 42 words


In [19]:
points = []

# FAQ points
for i, (vec, payload) in enumerate(zip(faq_vectors, faq_payloads)):
    points.append(
        PointStruct(
            id=i,
            vector=vec.tolist(),
            payload=payload
        )
    )

offset = len(points)

# Graph-node points (id continues after FAQ)
for j, (vec, payload) in enumerate(zip(graph_vectors, graph_payloads)):
    points.append(
        PointStruct(
            id=offset + j,
            vector=vec.tolist(),
            payload=payload
        )
    )

qdrant.upsert(collection_name="knowledge_space", points=points)
print(f"✅ Upserted {len(points)} total points into 'knowledge_space'")


✅ Upserted 45 total points into 'knowledge_space'


In [20]:
def classify_multi_intent(email_text: str, available_intents: list) -> list:
    """
    Classify multiple intents in an email using LLM.
    Returns list of intents (can be multiple).
    """
    intents_str = ", ".join(available_intents)
    
    prompt = f"""You are an email intent classifier.

Available intents (choose from these EXACT labels):
{intents_str}

Email to classify:
\"\"\"{email_text}\"\"\"

Instructions:
1. Identify ALL relevant intents from the list above
2. An email can have multiple intents (e.g., "share resume" = send_materials, "schedule meeting" = schedule)
3. Return ONLY a JSON array with exact labels from the list
4. Use underscores for multi-word intents (e.g., send_materials, not "send materials")

Examples:
- "Can you send me your resume?" → ["send_materials"]
- "Share your resume and let's schedule a call" → ["send_materials", "schedule"]
- "What time works for a meeting?" → ["schedule"]

Return ONLY valid JSON array, nothing else:
"""
    
    try:
        response = ollama.chat(
            model="llama3",
            messages=[{"role": "user", "content": prompt}]
        )
        
        content = response["message"]["content"].strip()
        
        # Debug: print what LLM returned
        print(f"🔍 LLM raw response: {content[:200]}...")
        
        # Try to extract JSON from markdown code blocks
        import json
        import re
        
        # Remove markdown code blocks if present
        content = re.sub(r'```json\s*', '', content)
        content = re.sub(r'```\s*', '', content)
        content = content.strip()
        
        try:
            intents = json.loads(content)
            if isinstance(intents, list):
                # Filter to only valid intents
                valid_intents = [i for i in intents if i in available_intents]
                if valid_intents:
                    print(f"✅ Classified intents: {valid_intents}")
                    return valid_intents
        except json.JSONDecodeError:
            # Try to extract array from text
            match = re.search(r'\[.*?\]', content)
            if match:
                try:
                    intents = json.loads(match.group())
                    if isinstance(intents, list):
                        valid_intents = [i for i in intents if i in available_intents]
                        if valid_intents:
                            print(f"✅ Classified intents: {valid_intents}")
                            return valid_intents
                except:
                    pass
        
        # Fallback: try to find intent names in the response
        for intent in available_intents:
            if intent.lower() in content.lower():
                print(f"✅ Classified intents (fallback): [{intent}]")
                return [intent]
        
        print("⚠️ Could not parse intents, using default")
        return ["general_inquiry"]
        
    except Exception as e:
        print(f"⚠️ Error classifying intent: {e}")
        return ["general_inquiry"]


def get_nodes_by_intents(intents: list, limit: int = 5) -> list:
    """
    Retrieve graph nodes related to the detected intents.
    This is the key enhancement - direct intent-to-node lookup!
    INCLUDES RELATIONSHIPS from edges!
    """
    def get_relationships(node_name: str):
        """Extract relationship information from edges."""
        if node_name not in G:
            return {"outgoing": [], "incoming": []}
        
        outgoing = []
        for successor in G.successors(node_name):
            edge_data = G.get_edge_data(node_name, successor, {})
            outgoing.append({
                "node": successor,
                "relation": edge_data.get("relation", "CONNECTED"),
                "email_id": edge_data.get("email_id")
            })
        
        incoming = []
        for predecessor in G.predecessors(node_name):
            edge_data = G.get_edge_data(predecessor, node_name, {})
            incoming.append({
                "node": predecessor,
                "relation": edge_data.get("relation", "CONNECTED"),
                "email_id": edge_data.get("email_id")
            })
        
        return {"outgoing": outgoing, "incoming": incoming}
    
    nodes = []
    seen_names = set()
    
    for intent in intents:
        # Check if intent exists as a node in graph
        if intent in G:
            # Get node info
            node_data = G.nodes[intent]
            neighbors = list(G.successors(intent)) + list(G.predecessors(intent))
            relationships = get_relationships(intent)  # ← Get relationships!
            
            if intent not in seen_names:
                nodes.append({
                    "name": intent,
                    "type": node_data.get("type", "unknown"),
                    "neighbors": neighbors,
                    "relationships": relationships  # ← Include relationships!
                })
                seen_names.add(intent)
            
            # Also get connected nodes (topics, artifacts)
            for neighbor in neighbors:
                if neighbor not in seen_names and len(nodes) < limit:
                    neighbor_data = G.nodes[neighbor]
                    neighbor_neighbors = list(G.successors(neighbor)) + list(G.predecessors(neighbor))
                    neighbor_relationships = get_relationships(neighbor)  # ← Get relationships!
                    nodes.append({
                        "name": neighbor,
                        "type": neighbor_data.get("type", "unknown"),
                        "neighbors": neighbor_neighbors,
                        "relationships": neighbor_relationships  # ← Include relationships!
                    })
                    seen_names.add(neighbor)
    
    return nodes[:limit]


print("✅ classify_multi_intent and get_nodes_by_intents loaded!")


✅ classify_multi_intent and get_nodes_by_intents loaded!


In [21]:
# ✅ GMAIL-STYLE: Updated functions with style-based retrieval

def build_prompt_with_style(email_text, intent, faq_hits, graph_hits, expanded_graph_info, style_examples=None):
    """
    Build prompt with content context (FAQs, graph) and style context (Gmail-style).
    
    Args:
        style_examples: List of style examples from semantic search (Gmail approach)
    """
    faq_section = "\n".join([
        f"{i+1}. Q: {f['question']}\n   A: {f['answer']}"
        for i, f in enumerate(faq_hits)
    ]) or "None"

    graph_section = "\n".join([
        f"{i+1}. Node: {g['node_name']} (type={g['node_type']}), neighbors={g.get('neighbors', [])}"
        for i, g in enumerate(graph_hits)
    ]) or "None"

    expansion_section = "\n".join([
        f"{i+1}. {node} → {neighbors}"
        for i, (node, neighbors) in enumerate(expanded_graph_info.items())
    ]) or "None"

    # ✅ NEW: Style section (Gmail-style approach)
    style_section = ""
    if style_examples:
        style_section = "\n".join([
            f"{i+1}. Style Example {i+1} (Zubair's writing tone):\n   \"{s['reply_chunk'][:250]}...\""
            for i, s in enumerate(style_examples[:3])  # Show top 3 style examples
        ])
    else:
        style_section = "None"

    prompt = f"""
You are **Zubair**, a graduate student known for being polite, proactive, and clear in communication.

Your job is to draft a short, natural, and professional email reply.

Keep it warm but not overly formal — think of how a thoughtful student would respond to a professor, coordinator, or peer.

**IMPORTANT**: Match the writing style shown in the style examples below. These are examples of Zubair's actual writing tone. Use similar phrasing, level of formality, and structure.

---

✉️ **Incoming Email**
\"\"\"{email_text}\"\"\"

🎯 **Detected Intent**: {intent}

📘 **Relevant FAQs** (Content Context)
{faq_section}

🧩 **Graph Context** (Structured Relationships)
{graph_section}

🔗 **Related Concepts**
{expansion_section}

✍️ **Writing Style Examples** (Style Anchor - Match This Tone)
{style_section}

---

Write your reply:
- Match the tone and style from the examples above
- Use similar phrasing, level of formality, and structure
- Acknowledge the sender and context
- If an action is requested, confirm or ask a polite follow-up question
- Keep the reply under 120 words
- Do NOT invent facts — only use what's in context
"""
    return prompt

print("✅ build_prompt_with_style loaded!")


✅ build_prompt_with_style loaded!


In [22]:
# ✅ GMAIL-STYLE: Updated answer_email_enhanced_fixed with style retrieval

def answer_email_enhanced_fixed_with_style(email_text: str, top_k: int = 6, show_context: bool = True):
    """
    Enhanced email answering with Gmail-style approach:
    1. Multi-intent classification
    2. Separate FAQ search (Qdrant) - ALWAYS retrieves FAQs
    3. Intent-based graph retrieval (NetworkX) - NO Qdrant for graph nodes
    4. Graph expansion
    5. ✅ NEW: Style-based retrieval (semantic search for writing style)
    """
    
    # Step 1: Multi-intent classification
    intents = classify_multi_intent(email_text, list(unique_intents))
    primary_intent = intents[0] if intents else "general_inquiry"
    
    if show_context:
        print(f"🎯 Detected Intents: {intents}")
        print(f"   Primary: {primary_intent}\n")
    
    # Step 2: Embed query for vector search
    q_vec = embedder.encode([email_text])[0].tolist()

    # Step 3: Search Qdrant for FAQs ONLY
    try:
        from qdrant_client.models import Filter, FieldCondition, MatchValue
        faq_search_results = qdrant.query_points(
            collection_name="knowledge_space",
            query=q_vec,
            limit=top_k,
            query_filter=Filter(
                must=[FieldCondition(key="type", match=MatchValue(value="faq"))]
            )
        ).points
    except (AttributeError, TypeError, ImportError):
        # Fallback: search all and filter manually
        all_hits = qdrant.search(
            collection_name="knowledge_space",
            query_vector=q_vec,
            limit=top_k * 2,
            with_payload=True
        )
        faq_search_results = [h for h in all_hits if h.payload.get("type") == "faq"][:top_k]
    
    # Extract FAQ hits
    faq_hits = []
    for h in faq_search_results:
        p = h.payload
        if p.get("type") == "faq":
            faq_hits.append({"score": h.score, **p})

    if show_context:
        print(f"📊 FAQ Search Results:")
        print(f"   FAQ hits: {len(faq_hits)}\n")

    # Step 4: Intent-based graph retrieval (using NetworkX, NO Qdrant for graph nodes)
    intent_graph_nodes = get_nodes_by_intents(intents, limit=5)
    
    # Convert to graph hit format
    graph_hits = []
    for node in intent_graph_nodes:
        graph_hits.append({
            "score": 0.75,
            "node_name": node["name"],
            "node_type": node["type"],
            "neighbors": node["neighbors"],
            "relationships": node.get("relationships", {"outgoing": [], "incoming": []})
        })
    
    if show_context:
        print(f"🕸️ Intent-based Graph Retrieval:")
        print(f"   Total graph nodes: {len(graph_hits)}\n")

    # Step 5: Graph expansion
    expanded_graph_info = {}
    for g in graph_hits:
        node = g["node_name"]
        if node in G:
            neighbors = list(G.successors(node)) + list(G.predecessors(node))
            expanded_graph_info[node] = neighbors

    # ✅ NEW: Step 6 - Style-based retrieval (Gmail approach)
    # Semantic search for writing style chunks
    try:
        style_hits = qdrant.search(
            collection_name="writing_style",
            query_vector=q_vec,
            limit=3,  # Get top 3 style examples
            with_payload=True
        )
        
        style_examples = []
        for hit in style_hits:
            style_examples.append({
                "score": hit.score,
                "reply_chunk": hit.payload.get("reply_chunk", ""),
                "subject": hit.payload.get("subject", ""),
                "intent": hit.payload.get("intent", "")
            })
    except Exception as e:
        if show_context:
            print(f"⚠️ Style search error: {e}")
        style_examples = []

    if show_context:
        print(f"✍️ Style Retrieval (Gmail-style):")
        print(f"   Style examples: {len(style_examples)}")
        if style_examples:
            for i, s in enumerate(style_examples, 1):
                print(f"   {i}. [Score {s['score']:.3f}] {s['reply_chunk'][:80]}...")
        print()

    # Print detailed context
    if show_context:
        print("\n" + "="*70)
        print("🔍 RETRIEVED CONTEXT")
        print("="*70)
        
        print("\n📚 FAQ Chunks:")
        if faq_hits:
            for i, f in enumerate(faq_hits, 1):
                print(f"  {i}. [Score {f['score']:.3f}] Q: {f['question']}")
                print(f"     A: {f['answer'][:80]}...\n")
        else:
            print("  None\n")

        print("🕸️ Graph Nodes:")
        if graph_hits:
            for i, g in enumerate(graph_hits, 1):
                print(f"  {i}. [Score {g['score']:.3f}] {g['node_name']} (type: {g['node_type']})")
                neighbors_str = ", ".join(g.get('neighbors', [])[:5])
                print(f"     Neighbors: {neighbors_str}\n")
        else:
            print("  None\n")
        
        print("✍️ Style Examples:")
        if style_examples:
            for i, s in enumerate(style_examples, 1):
                print(f"  {i}. [Score {s['score']:.3f}] {s['reply_chunk'][:100]}...\n")
        else:
            print("  None\n")
        
        print("🔗 Expanded Graph Context:")
        if expanded_graph_info:
            for node, neighbors in list(expanded_graph_info.items())[:5]:
                print(f"  {node} → {', '.join(neighbors[:5])}")
        else:
            print("  None")
        
        print("="*70 + "\n")

    # Step 7: Build enhanced prompt with style examples
    prompt = build_prompt_with_style(
        email_text, 
        ", ".join(intents),
        faq_hits, 
        graph_hits, 
        expanded_graph_info,
        style_examples  # ✅ Pass style examples
    )

    # Step 8: Generate reply
    resp = ollama.chat(
        model="llama3",
        messages=[{"role": "user", "content": prompt}]
    )
    reply_text = resp["message"]["content"]

    # Step 9: Confidence scoring (use FAQ top score if available)
    top_score = faq_hits[0]["score"] if faq_hits else (graph_hits[0]["score"] if graph_hits else 0.0)
    auto_send = top_score > 0.85

    return {
        "intents": intents,
        "primary_intent": primary_intent,
        "reply": reply_text,
        "top_score": top_score,
        "auto_send": auto_send,
        "faq_used": faq_hits,
        "graph_used": graph_hits,
        "style_used": style_examples,  # ✅ Include style examples
        "graph_expansion": expanded_graph_info,
    }

print("✅ answer_email_enhanced_fixed_with_style loaded!")
print("📝 Usage: result = answer_email_enhanced_fixed_with_style(email_text)")
print("🆕 Features:")
print("   - Separate FAQ search (always retrieves FAQs)")
print("   - Intent-based graph retrieval (NetworkX only)")
print("   - ✅ Gmail-style semantic search for writing style")
print("   - Style examples included in prompt for tone matching")


✅ answer_email_enhanced_fixed_with_style loaded!
📝 Usage: result = answer_email_enhanced_fixed_with_style(email_text)
🆕 Features:
   - Separate FAQ search (always retrieves FAQs)
   - Intent-based graph retrieval (NetworkX only)
   - ✅ Gmail-style semantic search for writing style
   - Style examples included in prompt for tone matching


In [23]:
test_email = """Hi Zubair,
Thank you for reaching out. To proceed with your interest in the Advanced Analytics position at Google, kindly share your resume and provide the right time to connect with you for an online meeting.
Regards,
Arjun Das
Talent Acquisition
Google Inc."""

result = answer_email_enhanced_fixed_with_style(test_email)
print("\n" + "="*70)
print("📧 FINAL RESULT")
print("="*70)
print(f"Intents: {result['intents']}")
print(f"Confidence: {result['top_score']:.3f}")
print(f"Auto-send: {result['auto_send']}")
print(f"\n✍️ Generated Reply:")
print("-"*70)
print(result['reply'])
print("-"*70)

🔍 LLM raw response: ["send_materials", "schedule"]...
✅ Classified intents: ['send_materials', 'schedule']
🎯 Detected Intents: ['send_materials', 'schedule']
   Primary: send_materials

📊 FAQ Search Results:
   FAQ hits: 2

🕸️ Intent-based Graph Retrieval:
   Total graph nodes: 5

✍️ Style Retrieval (Gmail-style):
   Style examples: 3
   1. [Score 0.543] Hello Sarah,

Thank you for considering my application at BrightFuture. Please f...
   2. [Score 0.508] Hello Laura,

Thank you for considering my application to the Data Scientist rol...
   3. [Score 0.503] Dear Jane,

Thank you for reaching out! I'm excited about the opportunity at Tec...


🔍 RETRIEVED CONTEXT

📚 FAQ Chunks:
  1. [Score 0.422] Q: Can I see Zubair's Data Science resume?
     A: Yes, here is the link to Zubair's Data Science resume: https://drive.google.com/...

  2. [Score 0.399] Q: Can I see Zubair's Software Engineering resume?
     A: Yes, here is the link to Zubair's Software Engineering resume: https://drive.goo.

### The top similarity score measures how semantically close your input is to your best-matching knowledge chunk. It’s a confidence proxy for deciding whether the LLM’s reply is likely grounded in the right retrieved context.

In [24]:
test_email = (
    "Hi Zubair, I hope you’re doing well. Can I get your linkedin profile to connect?"
)

result = answer_email_enhanced_fixed_with_style(test_email)
print("\n" + "="*70)
print("📧 FINAL RESULT")
print("="*70)
print(f"Intents: {result['intents']}")
print(f"Confidence: {result['top_score']:.3f}")
print(f"Auto-send: {result['auto_send']}")
print(f"\n✍️ Generated Reply:")
print("-"*70)
print(result['reply'])
print("-"*70)

🔍 LLM raw response: ["request_info"]...
✅ Classified intents: ['request_info']
🎯 Detected Intents: ['request_info']
   Primary: request_info

📊 FAQ Search Results:
   FAQ hits: 5

🕸️ Intent-based Graph Retrieval:
   Total graph nodes: 4

✍️ Style Retrieval (Gmail-style):
   Style examples: 3
   1. [Score 0.725] Dear Sarah, Thank you for considering my application! I have attached a link to ...
   2. [Score 0.690] Dear Michael, Thank you for reaching out! My LinkedIn profile is here: https://w...
   3. [Score 0.645] Dear Jane, Thank you for considering my application! My LinkedIn profile can be ...


🔍 RETRIEVED CONTEXT

📚 FAQ Chunks:
  1. [Score 0.605] Q: What is Zubair's LinkedIn profile?
     A: https://www.linkedin.com/in/zubair-atha/...

  2. [Score 0.413] Q: How can I contact Zubair for a call?
     A: You can reach Zubair by phone at +16463925601 or schedule a meeting via Calendly...

  3. [Score 0.409] Q: What is Zubair's professional background?
     A: Zubair is a graduate stu

In [25]:
test_email = (
    "Hi Zubair,"
    "Hope you're doing well. I wanted to check if our meeting scheduled for tomorrow is still on."
    "If not, could you suggest another time that works for you?"
    "Thanks,"
    "Alex"
)

result = answer_email_enhanced_fixed_with_style(test_email)
print("\n" + "="*70)
print("📧 FINAL RESULT")
print("="*70)
print(f"Intents: {result['intents']}")
print(f"Confidence: {result['top_score']:.3f}")
print(f"Auto-send: {result['auto_send']}")
print(f"\n✍️ Generated Reply:")
print("-"*70)
print(result['reply'])
print("-"*70)

🔍 LLM raw response: ["confirm", "schedule"]...
✅ Classified intents: ['confirm', 'schedule']
🎯 Detected Intents: ['confirm', 'schedule']
   Primary: confirm

📊 FAQ Search Results:
   FAQ hits: 5

🕸️ Intent-based Graph Retrieval:
   Total graph nodes: 5

✍️ Style Retrieval (Gmail-style):
   Style examples: 3
   1. [Score 0.656] Hello Mike, I'm available Monday through Thursday between 10 AM and 4 PM. Can yo...
   2. [Score 0.632] Hi John,

Thanks for reaching out! I'm free on Mondays and Wednesdays between 9 ...
   3. [Score 0.630] Hi Professor Wilson,
I am available on Wednesday at 3 PM and will be ready for o...


🔍 RETRIEVED CONTEXT

📚 FAQ Chunks:
  1. [Score 0.560] Q: How can I schedule a meeting with Zubair?
     A: Please use my Calendly link https://calendly.com/za2366-columbia/30min to book a...

  2. [Score 0.489] Q: What are Zubair's preferred meeting times?
     A: Zubair prefers Wednesday or Thursday between 10 AM and 2 PM Eastern Time. You ca...

  3. [Score 0.472] Q: What 

In [26]:
test_email = (
    "Hi Zubair,"
    "Hope you're doing well. I wanted to check if our meeting scheduled for tomorrow is still on."
    "If not, could you suggest another time that works for you?"
    "Thanks,"
    "Alex"
)

result = answer_email_enhanced_fixed_with_style(test_email)
print("\n" + "="*70)
print("📧 FINAL RESULT")
print("="*70)
print(f"Intents: {result['intents']}")
print(f"Confidence: {result['top_score']:.3f}")
print(f"Auto-send: {result['auto_send']}")
print(f"\n✍️ Generated Reply:")
print("-"*70)
print(result['reply'])
print("-"*70)

🔍 LLM raw response: ["request_info", "schedule"]...
✅ Classified intents: ['request_info', 'schedule']
🎯 Detected Intents: ['request_info', 'schedule']
   Primary: request_info

📊 FAQ Search Results:
   FAQ hits: 5

🕸️ Intent-based Graph Retrieval:
   Total graph nodes: 5

✍️ Style Retrieval (Gmail-style):
   Style examples: 3
   1. [Score 0.656] Hello Mike, I'm available Monday through Thursday between 10 AM and 4 PM. Can yo...
   2. [Score 0.632] Hi John,

Thanks for reaching out! I'm free on Mondays and Wednesdays between 9 ...
   3. [Score 0.630] Hi Professor Wilson,
I am available on Wednesday at 3 PM and will be ready for o...


🔍 RETRIEVED CONTEXT

📚 FAQ Chunks:
  1. [Score 0.560] Q: How can I schedule a meeting with Zubair?
     A: Please use my Calendly link https://calendly.com/za2366-columbia/30min to book a...

  2. [Score 0.489] Q: What are Zubair's preferred meeting times?
     A: Zubair prefers Wednesday or Thursday between 10 AM and 2 PM Eastern Time. You ca...

  3. [S

In [27]:

test_email = (
    "Hi Zubair,"
    "Did you get a chance to send the updated project slides to Prof. Rivera?"
    "He emailed me this morning asking for them, so I just wanted to confirm."
    "Let me know!"
    "Best,"
    "Alex"
)

result = answer_email_enhanced_fixed_with_style(test_email)
print("\n" + "="*70)
print("📧 FINAL RESULT")
print("="*70)
print(f"Intents: {result['intents']}")
print(f"Confidence: {result['top_score']:.3f}")
print(f"Auto-send: {result['auto_send']}")
print(f"\n✍️ Generated Reply:")
print("-"*70)
print(result['reply'])
print("-"*70)

🔍 LLM raw response: ["confirm"]...
✅ Classified intents: ['confirm']
🎯 Detected Intents: ['confirm']
   Primary: confirm

📊 FAQ Search Results:
   FAQ hits: 6

🕸️ Intent-based Graph Retrieval:
   Total graph nodes: 3

✍️ Style Retrieval (Gmail-style):
   Style examples: 3
   1. [Score 0.460] Hi Alice, your slides are very clear and engaging. Some areas to consider: add m...
   2. [Score 0.434] Dear Professor Chen,

I would be glad to discuss our ongoing project. Please che...
   3. [Score 0.418] Hi Mark,

Thanks for confirming. I'll bring all relevant documents to our meetin...


🔍 RETRIEVED CONTEXT

📚 FAQ Chunks:
  1. [Score 0.398] Q: Can I see Zubair's Data Science resume?
     A: Yes, here is the link to Zubair's Data Science resume: https://drive.google.com/...

  2. [Score 0.386] Q: Can I see Zubair's Software Engineering resume?
     A: Yes, here is the link to Zubair's Software Engineering resume: https://drive.goo...

  3. [Score 0.371] Q: Where can I find Zubair's work samples

In [29]:

test_email = (
    "Hello Zubair,"

"We're currently on the lookout for skilled individuals in machine learning, and your profile caught our eye. Would you mind sharing your resume with us? We'd love to explore potential opportunities together."
"Best,"
"Salman Khan"
"Bhai Dynamics"

)

result = answer_email_enhanced_fixed_with_style(test_email)
print("\n" + "="*70)
print("📧 FINAL RESULT")
print("="*70)
print(f"Intents: {result['intents']}")
print(f"Confidence: {result['top_score']:.3f}")
print(f"Auto-send: {result['auto_send']}")
print(f"\n✍️ Generated Reply:")
print("-"*70)
print(result['reply'])
print("-"*70)

🔍 LLM raw response: ["request_info", "send_materials"]...
✅ Classified intents: ['request_info', 'send_materials']
🎯 Detected Intents: ['request_info', 'send_materials']
   Primary: request_info

📊 FAQ Search Results:
   FAQ hits: 6

🕸️ Intent-based Graph Retrieval:
   Total graph nodes: 5

✍️ Style Retrieval (Gmail-style):
   Style examples: 3
   1. [Score 0.550] Hello Laura,

Thank you for considering my application to the Data Scientist rol...
   2. [Score 0.546] Hello Sarah,

Thank you for considering my application at BrightFuture. Please f...
   3. [Score 0.538] Hello Alex, Thank you for reaching out. Attached is my Software Engineering Resu...


🔍 RETRIEVED CONTEXT

📚 FAQ Chunks:
  1. [Score 0.469] Q: Can I see Zubair's Software Engineering resume?
     A: Yes, here is the link to Zubair's Software Engineering resume: https://drive.goo...

  2. [Score 0.468] Q: Can I see Zubair's Data Science resume?
     A: Yes, here is the link to Zubair's Data Science resume: https://drive.go

In [30]:

test_email = (
    "Dear Zubair"
"I trust this email finds you in good health. My name is Zohan Sharma, and I am a software engineer exploring opportunities to delve into deep learning. Your accomplishments in this field have inspired me, and I am reaching out to seek your guidance on the best learning path and potential projects to undertake. Your insights would mean a lot to me."
"Best regards, "
"Zohan"

)

result = answer_email_enhanced_fixed_with_style(test_email)
print("\n" + "="*70)
print("📧 FINAL RESULT")
print("="*70)
print(f"Intents: {result['intents']}")
print(f"Confidence: {result['top_score']:.3f}")
print(f"Auto-send: {result['auto_send']}")
print(f"\n✍️ Generated Reply:")
print("-"*70)
print(result['reply'])
print("-"*70)

🔍 LLM raw response: ["request_info"]...
✅ Classified intents: ['request_info']
🎯 Detected Intents: ['request_info']
   Primary: request_info

📊 FAQ Search Results:
   FAQ hits: 6

🕸️ Intent-based Graph Retrieval:
   Total graph nodes: 4

✍️ Style Retrieval (Gmail-style):
   Style examples: 3
   1. [Score 0.487] Hi Team,

Absolutely, I'd love to collaborate! My GitHub repository at https://g...
   2. [Score 0.458] Hello Emily,

Thank you for considering my application at Innovatech Solutions. ...
   3. [Score 0.417] Hello Sarah,

Thank you for considering my application at BrightFuture. Please f...


🔍 RETRIEVED CONTEXT

📚 FAQ Chunks:
  1. [Score 0.343] Q: What is Zubair's professional background?
     A: Zubair is a graduate student with expertise in both Data Science and Software En...

  2. [Score 0.292] Q: What materials can Zubair share?
     A: Zubair can share his Data Science resume, Software Engineering resume, LinkedIn ...

  3. [Score 0.240] Q: Can I see Zubair's Software Eng